# 4-lang `cc_bbox_blur` initial trial

Does the EN/ZH winner **`cc_bbox_blur`** transfer to KO and JA?

For each partner `L ∈ {zh, ko, ja}`:

| Attack | Boxes | Score |
|--------|-------|-------|
| `uni_en` | EN + EN | EN + L |
| `uni_l` | L + L | EN + L |
| `multi` | EN + L | EN + L |

Defense: **EN ∩ L** Attn-last → CC top-2 + bbox snap → Gaussian blur fill.
Tune threshold on n=100 (max EN attacked acc), then **enforce thr ≥ 0.95** for the full
n=1000 run. Geometry: `FONT_SIZE=24`, `NUM_BOXES=2`, frozen `attack_pos`.


In [1]:
import importlib.util, subprocess, sys

# Only pip-install missing packages — full resolve can take minutes every run.
_REQUIRED = [
    ('open_clip', 'open_clip_torch'),
    ('transformers', 'transformers'),
    ('datasets', 'datasets'),
    ('matplotlib', 'matplotlib'),
    ('PIL', 'Pillow'),
    ('scipy', 'scipy'),
]
_missing = [pip for mod, pip in _REQUIRED if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing missing:', _missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
else:
    print('Deps already installed — skipping pip.')


Deps already installed — skipping pip.


In [2]:
import os, platform, random, json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import (
    ChineseCLIPModel, ChineseCLIPProcessor, AutoModel, AutoProcessor,
)
from scipy import ndimage

os.makedirs('results', exist_ok=True)

assert torch.cuda.is_available(), 'CUDA required — install a CUDA build of PyTorch'
DEVICE = 'cuda'
print('Device:', DEVICE, torch.cuda.get_device_name(0))

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
BLUR_RADIUS  = 12
THRESHOLDS   = [0.75, 0.80, 0.85, 0.90, 0.95]
PARTNER_LANGS = ['zh', 'ko', 'ja']
ATTACKS = ['uni_en', 'uni_l', 'multi']

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'zh': ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车'],
    'ko': ['비행기','자동차','새','고양이','사슴','개','개구리','말','배','트럭'],
    'ja': ['飛行機','自動車','鳥','猫','鹿','犬','カエル','馬','船','トラック'],
}
TMPL = {
    'en': 'a photo of a {}.',
    'zh': '一张{}的照片。',
    'ko': '{}의 사진.',
    'ja': '{}の写真。',
}
ALL_LANGS = ['en', 'zh', 'ko', 'ja']


Device: cuda NVIDIA GeForce RTX 5070 Ti


## 1. Models


In [3]:
def _clip_feat(out):
    if torch.is_tensor(out):
        return out
    if getattr(out, 'pooler_output', None) is not None:
        return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    backend = 'open_clip'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms(
            'ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    backend = 'hf_vision'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'],
                                       token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

class KoCLIP:
    lang = 'ko'
    backend = 'hf_vision'
    def __init__(self):
        self.m = AutoModel.from_pretrained(
            'Bingsu/clip-vit-base-patch32-ko',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = AutoProcessor.from_pretrained('Bingsu/clip-vit-base-patch32-ko')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['ko'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'])
        return F.normalize(_clip_feat(out), dim=-1)

class JaCLIP:
    lang = 'ja'
    backend = 'open_clip'
    def __init__(self):
        mid = 'hf-hub:llm-jp/llm-jp-clip-vit-base-patch16'
        self.m, _, self.pp = open_clip.create_model_and_transforms(mid)
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer(mid)
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['ja'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

def classify_batch(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP, 'ko': KoCLIP, 'ja': JaCLIP}
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in ALL_LANGS}
print('Models ready.')


Loading en... 

C:\Users\dev account\AppData\Local\Programs\Python\Python313\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.2s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.6s
Loading ko... 

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

6.2s
Loading ja... 

3.6s


Models ready.


## 2. Dataset + dual-box attack builder


In [4]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

attack_pos = _saved['attack_pos']
assert len(attack_pos['en']) == len(idx) and len(attack_pos['l']) == len(idx)

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

all_idx  = np.arange(len(clean_224))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean_224)} images; tune subset = {len(tune_idx)}')
print(f"Attack positions: frozen from sample JSON (ref {attack_pos['ref_bw']}x{attack_pos['ref_bh']})")

_FONT_CACHE = {}

def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk = os.path.join(wf, 'msyh.ttc')
        lat = os.path.join(wf, 'arial.ttf')
        ko  = os.path.join(wf, 'malgun.ttf')
        if not os.path.isfile(ko):
            ko = cjk
        return cjk, lat, ko
    cjk = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
    lat = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    if not os.path.isfile(cjk):
        cjk = '/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf'
    return cjk, lat, cjk

_CJK_FONT, _LAT_FONT, _KO_FONT = _font_paths()

def _font_for_lang(lang):
    if lang == 'en':
        return _LAT_FONT
    if lang == 'ko':
        return _KO_FONT
    return _CJK_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:
            _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except Exception:
            _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _clamp_xy(xy, bw, bh):
    x, y = int(xy[0]), int(xy[1])
    x = max(0, min(x, max(0, DISPLAY_SIZE - bw)))
    y = max(0, min(y, max(0, DISPLAY_SIZE - bh)))
    return x, y

def draw_dual_box(img, word0, lang0, word1, lang1, img_idx, already_224=False):
    """Place boxes at frozen EN/L anchors from the sample JSON."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    xy0 = attack_pos['en'][int(img_idx)]
    xy1 = attack_pos['l'][int(img_idx)]
    for word, lang, xy in [(word0, lang0, xy0), (word1, lang1, xy1)]:
        font = _get_font(_font_for_lang(lang))
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rx, ry = _clamp_xy(xy, bw, bh)
        draw.rectangle([rx, ry, rx + bw, ry + bh], fill='white')
        draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)
    return img

def build_attack(attack, L):
    out = []
    for i in range(len(clean_224)):
        t = int(target[i])
        en_w = CLASSES['en'][t]
        l_w  = CLASSES[L][t]
        if attack == 'uni_en':
            img = draw_dual_box(clean_224[i], en_w, 'en', en_w, 'en', i, True)
        elif attack == 'uni_l':
            img = draw_dual_box(clean_224[i], l_w, L, l_w, L, i, True)
        else:
            img = draw_dual_box(clean_224[i], en_w, 'en', l_w, L, i, True)
        out.append(img)
    return out

_strip = Image.new('RGB', (DISPLAY_SIZE * 4, 80), (240, 240, 240))
_sd = ImageDraw.Draw(_strip)
for j, (lang, word) in enumerate([
    ('en', 'airplane'), ('zh', '飞机'), ('ko', '비행기'), ('ja', '飛行機')
]):
    f = _get_font(_font_for_lang(lang), 28)
    _sd.text((j * DISPLAY_SIZE + 10, 20), f'{lang}: {word}', fill='black', font=f)
_strip.save('results/font_check.png')
print('Fonts: LAT=', _LAT_FONT, 'CJK=', _CJK_FONT, 'KO=', _KO_FONT)
print('Saved results/font_check.png')

clean_acc = {
    ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean())
    for ml in ALL_LANGS
}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})


Loaded 1000 images; tune subset = 100
Attack positions: frozen from sample JSON (ref 131x44)
Fonts: LAT= C:\WINDOWS\Fonts\arial.ttf CJK= C:\WINDOWS\Fonts\msyh.ttc KO= C:\WINDOWS\Fonts\malgun.ttf
Saved results/font_check.png


Clean acc: {'en': '85.9%', 'zh': '91.4%', 'ko': '89.6%', 'ja': '92.5%'}


## 3. Attn-last saliency (EN/JA open_clip, ZH/KO HF vision)


In [5]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min()
    mx = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def _make_openclip_hook(collector):
    def hook(module, inputs, output):
        q_in = inputs[0]
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()
        n_head = module.num_heads
        hd = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            attn = attn.softmax(dim=-1)
        collector.append(attn[0].detach().cpu())
    return hook

def _build_attn_cam(all_attns, variant='last'):
    a = all_attns[-1]
    cls_row = a.mean(0)[0, 1:]
    if variant == 'rollout':
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for att in all_attns:
            a_r = 0.5 * att.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

def classify_and_attn(lang, pil_img, variant='last'):
    # Return (pred, cam) for any of the four models.
    wrapper = models[lang]
    if wrapper.backend == 'open_clip':
        x = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
        collector = []
        handles = [
            rb.attn.register_forward_hook(_make_openclip_hook(collector))
            for rb in wrapper.m.visual.transformer.resblocks
        ]
        with torch.no_grad():
            feat = wrapper.m.visual(x)
            imf = F.normalize(feat, dim=-1)
            pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
        for h in handles:
            h.remove()
        return pred, _build_attn_cam(collector, variant)

    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        if hasattr(wrapper.m, 'visual_projection'):
            proj = wrapper.m.visual_projection(vis_out.pooler_output)
        else:
            proj = vis_out.pooler_output
        imf = F.normalize(proj, dim=-1)
        pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
    attns = [a[0].cpu() for a in vis_out.attentions]
    return pred, _build_attn_cam(attns, variant)

for lang in ALL_LANGS:
    p, cam = classify_and_attn(lang, clean_224[0], 'last')
    print(f'  {lang}: pred={p} cam={cam.shape}')
print('Saliency ready.')


  en: pred=7 cam=(7, 7)
  zh: pred=5 cam=(14, 14)
  ko: pred=7 cam=(7, 7)
  ja: pred=7 cam=(14, 14)
Saliency ready.


## 4. Masking helpers (`cc_bbox_blur`)


In [6]:
def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2, :-2] | pad[:-2, 1:-1] | pad[:-2, 2:] |
             pad[1:-1, :-2] | pad[1:-1, 1:-1] | pad[1:-1, 2:] |
             pad[2:, :-2] | pad[2:, 1:-1] | pad[2:, 2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0:
        mask = dilate_mask(mask, iterations=dilate)
    return mask

def filter_mask_components(mask, top_k=2, bbox_snap=False):
    labeled, n = ndimage.label(mask.astype(bool))
    if n == 0:
        return mask.astype(bool)
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    keep = set(np.argsort(sizes)[::-1][:top_k] + 1)
    out = np.zeros_like(mask, dtype=bool)
    for i in keep:
        comp = labeled == i
        if bbox_snap:
            ys, xs = np.where(comp)
            out[ys.min():ys.max() + 1, xs.min():xs.max() + 1] = True
        else:
            out |= comp
    return out

def apply_mask(pil_img, mask, fill='blur'):
    arr = np.array(pil_img.convert('RGB'))
    m = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8) * 255).resize(
            arr.shape[1::-1], Image.NEAREST)) > 127
    out = arr.copy()
    if fill == 'blur':
        blurred = np.array(Image.fromarray(arr).filter(
            ImageFilter.GaussianBlur(radius=BLUR_RADIUS)))
        out[m] = blurred[m]
    else:
        mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

def build_cc_bbox_blur_mask(cam_en, cam_l, threshold=0.95, dilate=3, top_k=2):
    inter = n_cam_intersection(cam_en, cam_l)
    mask = cam_to_mask(inter, threshold, dilate=dilate)
    return filter_mask_components(mask, top_k=top_k, bbox_snap=True)

print('Masking helpers ready.')


Masking helpers ready.


## 5. Defense runner (Option B: score EN + L)


In [7]:
def run_defense(L, images, indices, threshold=0.95, label=''):
    # Apply EN∩L cc_bbox_blur; return preds for EN and L.
    imgs_sub = [images[i] for i in indices]
    true_sub = true[indices]
    n = len(indices)
    print(f'  defense {label} L={L} n={n} thr={threshold}...', end=' ', flush=True)
    t0 = time.time()
    masked_imgs, coverages = [], []
    for img in imgs_sub:
        _, cam_en = classify_and_attn('en', img, 'last')
        _, cam_l = classify_and_attn(L, img, 'last')
        mask = build_cc_bbox_blur_mask(cam_en, cam_l, threshold=threshold)
        coverages.append(float(mask.mean()))
        masked_imgs.append(apply_mask(img, mask, fill='blur'))
    preds = {
        'en': classify_batch(models['en'], masked_imgs, CLASSES['en']),
        L: classify_batch(models[L], masked_imgs, CLASSES[L]),
    }
    elapsed = time.time() - t0
    print(f'{elapsed:.1f}s  cov={100 * np.mean(coverages):.1f}%')
    return {
        'preds': preds,
        'true': true_sub,
        'coverage': float(np.mean(coverages)),
        'time_s': elapsed,
    }

def metrics_for_pair(preds, true_sub, target_sub, L, baseline_acc, baseline_asr):
    out = {}
    for ml in ['en', L]:
        out[ml] = {
            'acc': float((preds[ml] == true_sub).mean()),
            'asr': float((preds[ml] == target_sub).mean()),
            'baseline_acc': baseline_acc[ml],
            'baseline_asr': baseline_asr[ml],
        }
    out['mean_acc'] = 0.5 * (out['en']['acc'] + out[L]['acc'])
    return out

print('Runner ready.')


Runner ready.


## 6. Main loop — tune n=100 → full n=1000

For each `(L, attack)`: build attack → baselines → tune thr on EN masked acc
(n=100) → freeze best thr → defended eval on full n=1000 + clean degradation.


In [8]:
tune_best_cfg = {}
comparison = {}

for L in PARTNER_LANGS:
    for attack in ATTACKS:
        cell_key = f'{L}/{attack}'
        print(f'\n========== {cell_key} ==========')
        out_dir = Path('results') / L / attack
        out_dir.mkdir(parents=True, exist_ok=True)

        attacked = build_attack(attack, L)
        score_langs = ['en', L]

        preds_atk = {ml: classify_batch(models[ml], attacked, CLASSES[ml]) for ml in score_langs}
        baseline_acc = {ml: float((preds_atk[ml] == true).mean()) for ml in score_langs}
        baseline_asr = {ml: float((preds_atk[ml] == target).mean()) for ml in score_langs}
        print('  attacked acc:', {k: f'{100*v:.1f}%' for k, v in baseline_acc.items()})
        print('  attacked ASR:', {k: f'{100*v:.1f}%' for k, v in baseline_asr.items()})

        # --- tune on n=100 ---
        best_acc, best_thr, best_cov = -1.0, 0.95, 0.0
        for thr in THRESHOLDS:
            res = run_defense(L, attacked, tune_idx, threshold=thr, label=f'tune thr={thr}')
            en_acc = float((res['preds']['en'] == res['true']).mean())
            l_acc = float((res['preds'][L] == res['true']).mean())
            print(f'    thr={thr} EN={100*en_acc:.1f}% {L.upper()}={100*l_acc:.1f}%')
            if en_acc > best_acc:
                best_acc, best_thr, best_cov = en_acc, thr, res['coverage']

        free_thr = best_thr
        best_thr = max(free_thr, 0.95)
        if best_thr != free_thr:
            # Recompute tune coverage at the floored thr for logging.
            floor_res = run_defense(
                L, attacked, tune_idx, threshold=best_thr, label=f'tune thr={best_thr} (floor)')
            best_cov = floor_res['coverage']
            best_acc = float((floor_res['preds']['en'] == floor_res['true']).mean())

        tune_best_cfg[cell_key] = {
            'threshold': best_thr,
            'threshold_free': free_thr,
            'tune_en_acc': best_acc,
            'tune_cov': best_cov,
            'L': L,
            'attack': attack,
        }
        print(
            f'  BEST thr={best_thr} (free={free_thr}) '
            f'tune EN={100*best_acc:.1f}% cov={100*best_cov:.1f}%'
        )

        # --- full n=1000 ---
        atk = run_defense(L, attacked, all_idx, threshold=best_thr, label='full-atk')
        cln = run_defense(L, clean_224, all_idx, threshold=best_thr, label='full-clean')
        defense = metrics_for_pair(atk['preds'], true, target, L, baseline_acc, baseline_asr)
        clean_deg = {
            ml: {
                'baseline_acc': clean_acc[ml],
                'masked_acc': float((cln['preds'][ml] == true).mean()),
                'delta_acc': float((cln['preds'][ml] == true).mean()) - clean_acc[ml],
            } for ml in score_langs
        }
        row = {
            'L': L,
            'attack': attack,
            'threshold': best_thr,
            'threshold_free': free_thr,
            'ran_full': True,
            'clean_acc': {ml: clean_acc[ml] for ml in score_langs},
            'baseline_acc': baseline_acc,
            'baseline_asr': baseline_asr,
            'defense': defense,
            'clean_deg': clean_deg,
            'coverage': atk['coverage'],
            'mean_acc': defense['mean_acc'],
            'mean_clean_delta': 0.5 * (
                clean_deg['en']['delta_acc'] + clean_deg[L]['delta_acc']),
            'cost': 4,
        }
        print(
            f'  FULL mean acc={100*defense["mean_acc"]:.1f}%  '
            f'clean d={100*row["mean_clean_delta"]:+.1f}pp  '
            f'cov={100*atk["coverage"]:.1f}%'
        )

        with open(out_dir / 'confusion_results.json', 'w', encoding='utf-8') as f:
            json.dump(row, f, indent=2)
        comparison[cell_key] = row

with open('results/tune_best_cfg.json', 'w', encoding='utf-8') as f:
    json.dump(tune_best_cfg, f, indent=2)
with open('results/comparison_summary.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2)
print('\nSaved results/tune_best_cfg.json and results/comparison_summary.json')



========== zh/uni_en ==========


  attacked acc: {'en': '3.8%', 'zh': '24.8%'}
  attacked ASR: {'en': '96.2%', 'zh': '74.3%'}
  defense tune thr=0.75 L=zh n=100 thr=0.75... 

2.1s  cov=46.7%
    thr=0.75 EN=33.0% ZH=45.0%
  defense tune thr=0.8 L=zh n=100 thr=0.8... 

2.1s  cov=34.3%
    thr=0.8 EN=43.0% ZH=51.0%
  defense tune thr=0.85 L=zh n=100 thr=0.85... 

2.1s  cov=23.4%
    thr=0.85 EN=39.0% ZH=55.0%
  defense tune thr=0.9 L=zh n=100 thr=0.9... 

...

2.2s  cov=15.9%
    thr=0.9 EN=62.0% JA=79.0%
  defense tune thr=0.95 L=ja n=100 thr=0.95... 

2.2s  cov=8.3%
    thr=0.95 EN=61.0% JA=79.0%
  defense tune thr=0.95 (floor) L=ja n=100 thr=0.95... 

2.2s  cov=8.3%
  BEST thr=0.95 (free=0.9) tune EN=61.0% cov=8.3%
  defense full-atk L=ja n=1000 thr=0.95... 

21.7s  cov=8.3%
  defense full-clean L=ja n=1000 thr=0.95... 

21.9s  cov=8.5%
  FULL mean acc=75.9%  clean d=-11.5pp  cov=8.3%

Saved results/tune_best_cfg.json and results/comparison_summary.json


## 7. Comparison figure


In [9]:
keys = list(comparison.keys())
n = len(keys)
fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.45 * n + 1.5)), sharey=True)

for ax, which in zip(axes, ['en', 'partner']):
    labels, atk_vals, def_vals = [], [], []
    for k in keys:
        row = comparison[k]
        L = row['L']
        ml = 'en' if which == 'en' else L
        labels.append(k)
        atk_vals.append(100 * row['baseline_acc'][ml])
        def_vals.append(100 * row['defense'][ml]['acc'])
    y = np.arange(len(labels))
    h = 0.35
    ax.barh(y - h / 2, atk_vals, h, label='attacked', color='#c44e52')
    ax.barh(y + h / 2, def_vals, h, label='defended', color='#4c72b0')
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    title = 'English CLIP' if which == 'en' else 'Partner CLIP (L)'
    ax.set_title(title)
    ax.set_xlim(0, 100)
    ax.legend(loc='lower right', fontsize=8)

fig.suptitle('4-lang cc_bbox_blur — attacked vs defended', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('results/final_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved results/final_comparison.png')

print('\n=== Summary ===')
hdr = f'{"cell":<16} {"thr":>5} {"atkEN":>7} {"defEN":>7} {"atkL":>7} {"defL":>7} {"mean":>7} {"cD":>7} {"full":>5}'
print(hdr)
for k, row in comparison.items():
    L = row['L']
    cdelta = row.get('mean_clean_delta')
    cdelta_s = f'{100*cdelta:+.1f}' if cdelta is not None else '  n/a'
    full_s = 'Y' if row.get('ran_full') else 'n'
    print(
        f'{k:<16} {row["threshold"]:5.2f} '
        f'{100*row["baseline_acc"]["en"]:6.1f}% '
        f'{100*row["defense"]["en"]["acc"]:6.1f}% '
        f'{100*row["baseline_acc"][L]:6.1f}% '
        f'{100*row["defense"][L]["acc"]:6.1f}% '
        f'{100*row["mean_acc"]:6.1f}% '
        f'{cdelta_s:>7} '
        f'{full_s:>5}'
    )


Saved results/final_comparison.png

=== Summary ===
cell               thr   atkEN   defEN    atkL    defL    mean      cD  full
zh/uni_en         0.95    3.8%   53.6%   24.8%   66.8%   60.2%    -1.5     Y
zh/uni_l          0.95   72.0%   72.1%   40.3%   62.4%   67.2%    -1.5     Y
zh/multi          0.95    4.5%   70.0%    6.4%   78.0%   74.0%    -1.5     Y
ko/uni_en         0.95    3.8%   59.9%   12.9%   67.6%   63.7%   -11.2     Y
ko/uni_l          0.95   70.0%   64.5%   78.2%   72.3%   68.4%   -11.2     Y
ko/multi          0.95    3.5%   65.4%   12.3%   74.4%   69.9%   -11.2     Y
ja/uni_en         0.95    3.8%   66.4%    3.2%   78.2%   72.3%   -11.5     Y
ja/uni_l          0.95   71.3%   64.9%   84.6%   80.7%   72.8%   -11.5     Y
ja/multi          0.95    4.1%   68.6%    5.4%   83.2%   75.9%   -11.5     Y
